# 08a. Add Failure Date Rows to Test Dataset

이 노트북은 `split_group_stratified/test.parquet`에 생략되어 있는 **고장 당일의 데이터(Raw Max Date)**를 `ST4000DM000_raw.parquet`로부터 매칭하여 디코딩 후 추가한 뒤 **새로운 파일명(`test_with_failure_date.parquet`)**으로 저장하는 작업을 수행합니다.

### 주요 로직:
1. `test.parquet`에서 고장난 디스크(최종 sequence의 마지막 day)들의 목록 및 마지막 날짜를 식별합니다.
2. 대용량 raw 파일에서 `failure == 1`인 행(고장 당일)을 고속 필터링 로드하고 비트 시프트 등을 통한 디코딩 로직(SMART 1, 7, 188)을 수행합니다.
3. 시리얼 번호와 고장일을 대조하여 raw SMART 피처 값들을 매칭합니다.
4. `v3.parquet`에서 각 디스크의 이전 최대 28일치 이력을 로드합니다.
5. 디코딩된 고장 당일 행과 28일 이력을 결합하여 DuckDB 윈도우 함수를 통해 `test.parquet`의 파생변수(28일 윈도우 피처 등)들을 정확하게 계산합니다.
6. 계산된 고장 당일 행을 기존 데이터에 병합 및 정렬 후 최종적으로 **`test_with_failure_date.parquet`** 파일로 저장합니다.

In [1]:
import os, gc, sys
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path

# 파일 경로 설정
TEST_PATH = r'../data/split_group_stratified/test.parquet'
OUTPUT_PATH = r'../data/split_group_stratified/test_with_failure_date.parquet'
RAW_PATH = r'../data/01_data_cleaning/ST4000DM000_raw.parquet'
V3_PATH = r'../data/01_data_cleaning/ST4000DM000_v3.parquet'
TMP_DIR = Path("../.tmp")
TMP_DIR.mkdir(exist_ok=True)

### 1. Test 데이터 로드 및 고장 시리얼 번호 추출

In [2]:
print("Loading test.parquet...")
df_test = pd.read_parquet(TEST_PATH)

# failure == 1을 가지는 시리얼 번호 목록
df_fail_test = df_test[df_test['failure'] == 1].copy()
failed_serials_test = df_fail_test['serial_number'].unique()

print(f"Total rows in test: {len(df_test):,}")
print(f"Total unique failed serials: {len(failed_serials_test):,}")

# base_serial 매핑 (_숫자 suffix 제거)
def strip_suffix(s):
    parts = s.rsplit('_', 1)
    if len(parts) == 2 and parts[1].isdigit():
        return parts[0]
    return s

serial_map = {s: strip_suffix(s) for s in failed_serials_test}
base_serials = list(set(serial_map.values()))
print(f"Total unique base serials: {len(base_serials):,}")

# 고장일 매핑
max_dates = df_fail_test.groupby('serial_number')['date'].max().reset_index()
max_dates.columns = ['serial_number_test', 'last_date']
max_dates['base_serial'] = max_dates['serial_number_test'].apply(strip_suffix)
max_dates['fail_date'] = pd.to_datetime(max_dates['last_date']) + pd.Timedelta(days=1)

Loading test.parquet...
Total rows in test: 15,695,096
Total unique failed serials: 1,127
Total unique base serials: 1,127


### 2. Raw 데이터에서 고장일 레코드 로드 및 디코딩

In [3]:
print("Loading failure day records from raw dataset...")
df_raw = pd.read_parquet(RAW_PATH, filters=[('failure', '==', 1)])
print(f"Loaded {len(df_raw):,} failure rows from raw.")

# Numeric Casting
numeric_raws = ['smart_1_raw','smart_7_raw','smart_188_raw',
                'smart_5_raw','smart_184_raw','smart_187_raw',
                'smart_197_raw','smart_198_raw','smart_199_raw',
                'smart_9_raw','smart_190_raw','smart_194_raw',
                'smart_241_raw','smart_242_raw']
for c in numeric_raws:
    if c in df_raw.columns:
        df_raw[c] = pd.to_numeric(df_raw[c], errors='coerce')

# 디코딩
if 'smart_1_raw' in df_raw.columns:
    v = df_raw['smart_1_raw'].fillna(0).astype(np.int64)
    df_raw['total_reads'] = (v & 0xFFFFFFFF).astype(float)
    df_raw.drop(columns=['smart_1_raw'], inplace=True)

if 'smart_7_raw' in df_raw.columns:
    s7 = pd.to_numeric(df_raw['smart_7_raw'], errors='coerce').fillna(0.0)
    v = s7.values.astype(np.int64)
    df_raw['seek_error_count'] = (v >> 32).astype(float)
    df_raw['total_seeks'] = (v & 0xFFFFFFFF).astype(float)
    df_raw.drop(columns=['smart_7_raw'], inplace=True)

if 'smart_188_raw' in df_raw.columns:
    s188 = pd.to_numeric(df_raw['smart_188_raw'], errors='coerce').fillna(0.0)
    v = s188.values.astype(np.int64)
    df_raw['timeout_total'] = (v & 0xFFFF).astype(float)
    df_raw['timeout_5s'] = ((v >> 16) & 0xFFFF).astype(float)
    df_raw.drop(columns=['smart_188_raw'], inplace=True)

# 온도 이상치 및 불필요 컬럼 정리
for c in ['smart_190_raw', 'smart_194_raw']:
    if c in df_raw.columns:
        df_raw.loc[df_raw[c] >= 100, c] = np.nan
        
df_raw.drop(columns=['smart_12_raw','smart_240_raw'], inplace=True, errors='ignore')
df_raw['date'] = pd.to_datetime(df_raw['date'])

# test에 존재하는 serial만 필터링
df_raw = df_raw[df_raw['serial_number'].isin(base_serials)].copy()
print(f"Base_serial matching failure rows: {len(df_raw)}")

Loading failure day records from raw dataset...
Loaded 5,791 failure rows from raw.
Base_serial matching failure rows: 1128


### 3. V3 데이터에서 고장 전 28일 이력 로드

In [4]:
print("Loading 28-day history from v3.parquet...")
v3_cols = ['serial_number','date','failure',
           'smart_5_raw','smart_184_raw','smart_187_raw','smart_197_raw',
           'smart_198_raw','smart_199_raw','smart_9_raw','smart_190_raw',
           'smart_194_raw','smart_241_raw','smart_242_raw',
           'total_reads','total_seeks','seek_error_count',
           'timeout_total','timeout_5s']

history_chunks = []
chunk_size = 200

for i in range(0, len(base_serials), chunk_size):
    batch = base_serials[i:i+chunk_size]
    serial_str = ", ".join([f"'{s}'" for s in batch])
    con = duckdb.connect()
    try:
        sql = f"""
        SELECT *
        FROM read_parquet('{V3_PATH.replace(chr(92),"/")}')
        WHERE serial_number IN ({serial_str})
          AND failure = 0
        QUALIFY ROW_NUMBER() OVER (PARTITION BY serial_number ORDER BY date DESC) <= 28
        """
        df_hist = con.execute(sql).df()
        existing_cols = [c for c in v3_cols if c in df_hist.columns]
        history_chunks.append(df_hist[existing_cols])
    finally:
        con.close()
    print(f"  Batch {i//chunk_size+1}/{(len(base_serials)+chunk_size-1)//chunk_size} done", end="\r")

df_hist_all = pd.concat(history_chunks, ignore_index=True)
df_hist_all['date'] = pd.to_datetime(df_hist_all['date'])
print(f"\nTotal history rows loaded: {len(df_hist_all):,}")

Loading 28-day history from v3.parquet...
  Batch 6/6 done
Total history rows loaded: 27,765


### 4. 윈도우 피처 계산 (DuckDB)

In [5]:
print("Calculating 28-day window features...")
FINAL_COLS = [
    "serial_number", "date", "failure",
    "s187_days_since_first", "smart_184_raw", "error_density_14d",
    "smart_187_raw", "smart_198_raw", "total_seeks_28d_asfd",
    "s187_28d_sum", "s5_days_since_first", "s199_days_since_last",
    "multi_error_count", "age_weighted_workload", "smart_242_raw",
    "smart_241_raw", "smart_9_raw", "total_seeks_diff", "s241_diff",
    "s194_28d_dai", "s194_28d_std", "s242_28d_dai",
    "total_reads_7d_asfd", "total_reads_7d_max", "s194_14d_max",
    "s194_28d_ewma", "s194_14d_std", "s190_28d_zscore",
    "s190_28d_ewma", "s190_28d_mean",
]

# 컬럼 통일
for c in v3_cols:
    if c not in df_hist_all.columns: df_hist_all[c] = 0.0
    if c not in df_raw.columns: df_raw[c] = 0.0

df_raw_slim = df_raw[v3_cols].copy()
df_raw_slim['failure'] = 1

result_rows = []

for idx, mrow in max_dates.iterrows():
    sn_test  = mrow['serial_number_test']
    base_sn  = mrow['base_serial']
    fail_dt  = mrow['fail_date']

    hist = df_hist_all[df_hist_all['serial_number'] == base_sn].copy().sort_values('date').tail(28)
    raw_row = df_raw_slim[df_raw_slim['serial_number'] == base_sn]
    
    if len(raw_row) == 0:
        continue
        
    raw_row = raw_row.iloc[0:1].copy()
    raw_row['date'] = fail_dt

    combined = pd.concat([hist, raw_row], ignore_index=True).sort_values('date').reset_index(drop=True)
    combined['date'] = pd.to_datetime(combined['date'])
    for c in v3_cols[3:]:
        combined[c] = pd.to_numeric(combined[c], errors='coerce').fillna(0.0)

    con = duckdb.connect()
    con.register("tbl", combined)
    try:
        sql = """
WITH raw AS (
    SELECT
        serial_number, failure,
        CAST(date AS DATE) AS dt,
        smart_5_raw, smart_184_raw, smart_187_raw, smart_197_raw, smart_198_raw,
        smart_9_raw, smart_199_raw, smart_241_raw, smart_242_raw,
        timeout_total, total_reads, total_seeks,
        smart_190_raw, smart_194_raw,
        smart_5_raw   - LAG(smart_5_raw)   OVER w AS s5_diff,
        smart_187_raw - LAG(smart_187_raw) OVER w AS s187_diff,
        smart_197_raw - LAG(smart_197_raw) OVER w AS s197_diff,
        smart_198_raw - LAG(smart_198_raw) OVER w AS s198_diff,
        smart_199_raw - LAG(smart_199_raw) OVER w AS s199_diff,
        smart_241_raw - LAG(smart_241_raw) OVER w AS s241_diff,
        smart_242_raw - LAG(smart_242_raw) OVER w AS s242_diff,
        timeout_total - LAG(timeout_total) OVER w AS timeout_total_diff,
        total_reads   - LAG(total_reads)   OVER w AS total_reads_diff,
        total_seeks   - LAG(total_seeks)   OVER w AS total_seeks_diff,
        smart_9_raw   - LAG(smart_9_raw)   OVER w AS s9_diff
    FROM tbl
    WINDOW w AS (PARTITION BY serial_number ORDER BY date)
),
d2 AS (
    SELECT *,
        COALESCE(total_seeks_diff,0) - LAG(COALESCE(total_seeks_diff,0)) OVER w AS d_seeks,
        COALESCE(total_reads_diff,0) - LAG(COALESCE(total_reads_diff,0)) OVER w AS d_reads,
        COALESCE(s242_diff,0)        - LAG(COALESCE(s242_diff,0))        OVER w AS d_s242
    FROM raw
    WINDOW w AS (PARTITION BY serial_number ORDER BY dt)
),
calc AS (
    SELECT
        serial_number, dt AS date, failure,
        CAST(smart_198_raw AS FLOAT)  AS smart_198_raw,
        CAST(smart_242_raw AS FLOAT)  AS smart_242_raw,
        CAST(smart_241_raw AS FLOAT)  AS smart_241_raw,
        CAST(smart_9_raw   AS FLOAT)  AS smart_9_raw,
        CAST(smart_184_raw AS FLOAT)  AS smart_184_raw,
        CAST(smart_187_raw AS FLOAT)  AS smart_187_raw,
        CAST(COALESCE(s5_diff,0)          AS FLOAT) AS s5_diff,
        CAST(COALESCE(s241_diff,0)        AS FLOAT) AS s241_diff,
        CAST(COALESCE(total_seeks_diff,0) AS FLOAT) AS total_seeks_diff,
        CAST(LN(ABS(COALESCE(s241_diff,0)+COALESCE(s242_diff,0))+1.0)
             * LN(smart_9_raw+1.0) AS FLOAT)         AS age_weighted_workload,
        CAST(
            CAST(COALESCE(s5_diff,0)>0 AS INT)
          + CAST(COALESCE(s187_diff,0)>0 AS INT)
          + CAST(COALESCE(s197_diff,0)>0 AS INT)
          + CAST(COALESCE(s198_diff,0)>0 AS INT)
          + CAST(COALESCE(timeout_total_diff,0)>0 AS INT)
        AS FLOAT)                                    AS multi_error_count,
        CAST(
            SUM(ABS(COALESCE(s5_diff,0))+ABS(COALESCE(s187_diff,0))
               +ABS(COALESCE(s197_diff,0))+ABS(COALESCE(s198_diff,0))
               +ABS(COALESCE(timeout_total_diff,0))) OVER w14
            / (SUM(GREATEST(0,COALESCE(s9_diff,0))) OVER w14 + 1.0)
        AS FLOAT)                                    AS error_density_14d,
        CAST(SUM(COALESCE(s187_diff,0)) OVER w28 AS FLOAT)          AS s187_28d_sum,
        CAST(SUM(ABS(COALESCE(d_seeks,0))) OVER w28 AS FLOAT)       AS total_seeks_28d_asfd,
        CAST(SUM(ABS(COALESCE(d_reads,0))) OVER w7  AS FLOAT)       AS total_reads_7d_asfd,
        CAST(COALESCE(STDDEV_SAMP(smart_194_raw) OVER w28,0.0) AS FLOAT) AS s194_28d_std,
        CAST(COALESCE(STDDEV_SAMP(smart_194_raw) OVER w14,0.0) AS FLOAT) AS s194_14d_std,
        CAST(AVG(smart_190_raw) OVER w28 AS FLOAT)                        AS s190_28d_mean,
        CAST(MAX(smart_194_raw) OVER w14 AS FLOAT)                        AS s194_14d_max,
        CAST(
            (COALESCE(s242_diff,0) - FIRST_VALUE(COALESCE(s242_diff,0)) OVER w28_dai)
            / NULLIF(COUNT(s242_diff) OVER w28_dai - 1, 0)
        AS FLOAT)                                    AS s242_28d_dai,
        CAST(
            (smart_194_raw - FIRST_VALUE(smart_194_raw) OVER w28_dai)
            / NULLIF(COUNT(smart_194_raw) OVER w28_dai - 1, 0)
        AS FLOAT)                                    AS s194_28d_dai,
        CAST(
            CASE WHEN COALESCE(STDDEV_SAMP(smart_190_raw) OVER w28,0.0) < 0.001 THEN 0.0
            ELSE (smart_190_raw - AVG(smart_190_raw) OVER w28)
                 / (COALESCE(STDDEV_SAMP(smart_190_raw) OVER w28,0.0) + 1e-5)
            END AS FLOAT)                            AS s190_28d_zscore,
        CAST(MAX(COALESCE(total_reads_diff,0)) OVER w7 AS FLOAT)    AS total_reads_7d_max,
        CASE WHEN MIN(CASE WHEN smart_5_raw>0 THEN dt END) OVER w IS NULL THEN -1
             ELSE CAST(date_diff('day',CAST(MIN(CASE WHEN smart_5_raw>0 THEN dt END) OVER w AS DATE),dt) AS INTEGER)
        END                                          AS s5_days_since_first,
        CASE WHEN MIN(CASE WHEN smart_187_raw>0 THEN dt END) OVER w IS NULL THEN -1
             ELSE CAST(date_diff('day',CAST(MIN(CASE WHEN smart_187_raw>0 THEN dt END) OVER w AS DATE),dt) AS INTEGER)
        END                                          AS s187_days_since_first,
        CASE WHEN MAX(CASE WHEN COALESCE(s199_diff,0)>0 THEN dt END) OVER w IS NULL THEN -1
             ELSE CAST(date_diff('day',CAST(MAX(CASE WHEN COALESCE(s199_diff,0)>0 THEN dt END) OVER w AS DATE),dt) AS INTEGER)
        END                                          AS s199_days_since_last,
        smart_190_raw AS _r190,
        smart_194_raw AS _r194
    FROM d2
    WINDOW
        w       AS (PARTITION BY serial_number ORDER BY dt),
        w7      AS (PARTITION BY serial_number ORDER BY dt ROWS BETWEEN 6  PRECEDING AND CURRENT ROW),
        w14     AS (PARTITION BY serial_number ORDER BY dt ROWS BETWEEN 13 PRECEDING AND CURRENT ROW),
        w28     AS (PARTITION BY serial_number ORDER BY dt ROWS BETWEEN 27 PRECEDING AND CURRENT ROW),
        w28_dai AS (PARTITION BY serial_number ORDER BY dt ROWS BETWEEN 27 PRECEDING AND CURRENT ROW)
)
SELECT * FROM calc
ORDER BY serial_number, date
        """
        df_out = con.execute(sql).df()
    finally:
        con.close()

    last = df_out[df_out['failure'] == 1].tail(1).copy()
    if last.empty:
        last = df_out.tail(1).copy()
        last['failure'] = 1

    for col_src, col_dst in [('_r190','s190_28d_ewma'), ('_r194','s194_28d_ewma')]:
        if col_src in df_out.columns:
            ewma_series = df_out[col_src].ewm(span=28, adjust=False).mean()
            last[col_dst] = float(ewma_series.iloc[-1])

    last.drop(columns=['_r190','_r194'], inplace=True, errors='ignore')
    last['serial_number'] = sn_test

    result_rows.append(last)

df_new = pd.concat(result_rows, ignore_index=True)

# FINAL_COLS 타입 및 구조 맞춤
for c in FINAL_COLS:
    if c not in df_new.columns:
        df_new[c] = 0.0

df_new = df_new[FINAL_COLS].copy()
df_new['date'] = pd.to_datetime(df_new['date'])
df_new['failure'] = df_new['failure'].astype('int64')

for c in df_test.select_dtypes('int32').columns:
    if c in df_new.columns: df_new[c] = df_new[c].astype('int32')
for c in df_test.select_dtypes('float32').columns:
    if c in df_new.columns: df_new[c] = df_new[c].astype('float32')

print(f"Successfully calculated features for {len(df_new):,} failure rows.")

Calculating 28-day window features...
Successfully calculated features for 1,127 failure rows.


### 5. 기존 Test 데이터에 추가 및 정렬

In [6]:
print("Appending new rows and sorting...")
df_test_updated = pd.concat([df_test, df_new], ignore_index=True)
df_test_updated.sort_values(['serial_number', 'date'], inplace=True)
df_test_updated.reset_index(drop=True, inplace=True)

print(f"Updated test dataset rows: {len(df_test_updated):,} (original: {len(df_test):,})")

Appending new rows and sorting...
Updated test dataset rows: 15,696,223 (original: 15,695,096)


### 6. 최종 결과물 저장 (새 파일로 분리)

In [7]:
print(f"Saving updated test dataset to {OUTPUT_PATH}...")
df_test_updated.to_parquet(OUTPUT_PATH, index=False, compression='zstd')
print("[OK] Saved successfully!")

Saving updated test dataset to ../data/split_group_stratified/test_with_failure_date.parquet...
[OK] Saved successfully!
